# Notebook 0 — Data Preparation (Instructor Only)

This notebook pre-computes everything students need for Notebooks 2 and 3.
**Students do not run this notebook.**

What it produces:
- Subfamily FASTA files in `data/subfamilies/`
- Per-subfamily alignments at two trimming levels (`-gt 0.1` and `-gt 0.5`)
- Per-subfamily FastTree trees
- Completed `data/species_classification.tsv`
- A PyMOL script template for optional structure visualization

In [ ]:
import os, subprocess, re
from collections import Counter, defaultdict

DATA = os.path.join('..', 'data')
SUB_DIR = os.path.join(DATA, 'subfamilies')
os.makedirs(SUB_DIR, exist_ok=True)

## 1. Load family tree and identify subfamilies

In [ ]:
from ete4 import PhyloTree

def read_fasta(path):
    seqs = {}
    header = None
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                header = line[1:].split()[0]
                seqs[header] = ''
            elif header:
                seqs[header] += line
    return seqs

def write_fasta(seqs, path):
    with open(path, 'w') as f:
        for header, seq in seqs.items():
            f.write(f'>{header}\n{seq}\n')

In [ ]:
FASTA = os.path.join(DATA, 'selection2_clustalo.fa')
TREE  = os.path.join(DATA, 'selection2.clustalo.gt01.fasttree.nwk')

all_seqs = read_fasta(FASTA)
print(f"Total sequences: {len(all_seqs)}")

t = PhyloTree(open(TREE).read(),
              sp_naming_function=lambda n: n.split('.')[0])
t.set_outgroup(t.get_midpoint_outgroup())
t.resolve_polytomy(descendants=True)
t.annotate_ncbi_taxa()
events = t.get_descendant_evol_events()
print(f"Leaves: {len(list(t.leaves()))}")
print(f"Duplication events: {sum(1 for e in events if e.etype == 'D')}")

In [ ]:
# Load gene name mapping
GNAME_FILE = os.path.join(DATA, 'selection2.clustalo.seqid2gname.tab')
seqid2gene = {}
if os.path.exists(GNAME_FILE):
    for line in open(GNAME_FILE):
        parts = line.strip().split('\t')
        if len(parts) >= 2:
            seqid2gene[parts[0]] = parts[1]
    print(f"Loaded {len(seqid2gene)} gene name mappings")

# Assign gene names to leaves
for leaf in t.leaves():
    gname = seqid2gene.get(leaf.name, '')
    leaf.add_prop('gene_name', gname)

## 2. Extract subfamilies using duplication events

Walk the tree from root. At each duplication node, the children
subtrees are different paralog groups. We label each leaf with
its subfamily based on the gene name mapping.

In [ ]:
# Strategy: use gene names to assign subfamilies where known,
# then propagate to unlabeled leaves via tree topology.

def guess_subfamily(gene_name):
    """Extract SLC26A* subfamily from a gene name string."""
    if not gene_name:
        return None
    text = gene_name.upper()
    m = re.search(r'S(?:LC)?26A(\d+)', text)
    if m:
        return f"SLC26A{m.group(1)}"
    if 'PRESTIN' in text or 'PRES' in text:
        return 'SLC26A5'
    if 'PENDRIN' in text or 'PDS' in text:
        return 'SLC26A4'
    return None

# Label leaves with known subfamily
labeled = 0
for leaf in t.leaves():
    gname = leaf.props.get('gene_name', '')
    sub = guess_subfamily(gname)
    if sub:
        leaf.add_prop('subfamily', sub)
        labeled += 1
    else:
        leaf.add_prop('subfamily', 'unknown')

print(f"Labeled {labeled}/{len(list(t.leaves()))} leaves from gene names")

In [ ]:
# Group leaves by subfamily
subfamily_seqs = defaultdict(dict)
for leaf in t.leaves():
    sub = leaf.props.get('subfamily', 'unknown')
    # Get ungapped sequence
    if leaf.name in all_seqs:
        ungapped = all_seqs[leaf.name].replace('-', '')
        subfamily_seqs[sub][leaf.name] = ungapped

print(f"\n{'Subfamily':<15s} {'Sequences':>10s}")
print('-' * 30)
for sub in sorted(subfamily_seqs):
    print(f"  {sub:<15s} {len(subfamily_seqs[sub]):>8d}")

In [ ]:
# Write subfamily FASTA files (ungapped, for re-alignment)
for sub, seqs in subfamily_seqs.items():
    if len(seqs) < 4:
        print(f"  Skipping {sub}: only {len(seqs)} sequences")
        continue
    out_path = os.path.join(SUB_DIR, f'{sub}.fa')
    write_fasta(seqs, out_path)
    print(f"  {sub}: {len(seqs)} sequences → {out_path}")

## 3. Align and trim each subfamily

In [ ]:
def run_cmd(cmd, label=""):
    """Run a shell command, print on failure."""
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"  ⚠ {label} failed: {result.stderr[:200]}")
    return result.returncode == 0

In [ ]:
for sub_file in sorted(os.listdir(SUB_DIR)):
    if not sub_file.endswith('.fa'):
        continue
    sub = sub_file.replace('.fa', '')
    fa = os.path.join(SUB_DIR, sub_file)

    aln   = os.path.join(SUB_DIR, f'{sub}.aln.fa')
    trim1 = os.path.join(SUB_DIR, f'{sub}.trim.fa')       # -gt 0.1 (for analysis)
    trim5 = os.path.join(SUB_DIR, f'{sub}.trim05.fa')      # -gt 0.5 (for visualization)
    tree  = os.path.join(SUB_DIR, f'{sub}.nwk')

    print(f"\n--- {sub} ---")

    # MAFFT alignment
    if not os.path.exists(aln):
        run_cmd(f'mafft --auto {fa} > {aln} 2>/dev/null', 'MAFFT')
        print(f"  MAFFT → {aln}")
    else:
        print(f"  MAFFT: exists")

    # trimAl -gt 0.1 (light trim, for analysis)
    if not os.path.exists(trim1):
        run_cmd(f'trimal -in {aln} -out {trim1} -gt 0.1', 'trimAl-gt01')
        print(f"  trimAl -gt 0.1 → {trim1}")
    else:
        print(f"  trimAl -gt 0.1: exists")

    # trimAl -gt 0.5 (strict trim, for visualization)
    if not os.path.exists(trim5):
        run_cmd(f'trimal -in {aln} -out {trim5} -gt 0.5', 'trimAl-gt05')
        print(f"  trimAl -gt 0.5 → {trim5}")
    else:
        print(f"  trimAl -gt 0.5: exists")

    # FastTree
    if not os.path.exists(tree):
        run_cmd(f'FastTree -lg -quiet < {trim1} > {tree} 2>/dev/null', 'FastTree')
        print(f"  FastTree → {tree}")
    else:
        print(f"  FastTree: exists")

## 4. Species classification

Print all species with taxonomy info so you can complete the TSV.

In [ ]:
taxid_counts = Counter(leaf.name.split('.')[0] for leaf in t.leaves())

print(f"{'TaxID':<10s} {'Species':<35s} {'Order':<20s} {'Genes':>5s}")
print('-' * 75)
for taxid, count in sorted(taxid_counts.items(), key=lambda x: -x[1]):
    leaf = next(l for l in t.leaves() if l.name.startswith(taxid + '.'))
    sci = leaf.props.get('sci_name', '?')
    lineage = leaf.props.get('named_lineage', [])
    order = '?'
    for name in lineage:
        if name in ('Chiroptera', 'Cetacea', 'Artiodactyla', 'Primates',
                     'Rodentia', 'Carnivora', 'Perissodactyla',
                     'Didelphimorphia', 'Dasyuromorphia', 'Proboscidea',
                     'Lagomorpha', 'Scandentia', 'Eulipotyphla'):
            order = name
            break
    print(f"  {taxid:<10s} {sci:<35s} {order:<20s} {count:>5d}")

Use the output above to complete `data/species_classification.tsv`.
Format: `taxid\tspecies\tgroup\techolocates\tnotes`

Groups: `echolocating_bat`, `non_echolocating_bat`,
`echolocating_cetacean`, `non_echolocating_cetacean`, `outgroup`

All Chiroptera except Pteropodidae (fruit bats) echolocate.
All Odontoceti (toothed whales) echolocate; Mysticeti (baleen) do not.

## 5. PyMOL script template

Human prestin PDB: **7S8X** (Bavi et al. 2021)

In [ ]:
pymol_template = """# Prestin convergent residues — PyMOL visualization
# Run: pymol prestin_convergent.pml
#
# Replace RESIDUE_LIST with actual convergent site positions
# (mapped to human prestin sequence numbering)

fetch 7S8X, prestin
remove solvent
remove chain B

hide everything
show cartoon, prestin
set cartoon_transparency, 0.3

# Color by domain (approximate boundaries)
color gray80, prestin
color tv_red, prestin and resi 81-505     # TMD
color tv_orange, prestin and resi 506-530  # Linker
color tv_blue, prestin and resi 531-744    # STAS

# Convergent residues (fill in from NB03 results)
# select convergent, prestin and resi RESIDUE_LIST
# show spheres, convergent and name CA
# color yellow, convergent and name CA

bg_color white
orient prestin
"""

pml_path = os.path.join(DATA, 'prestin_convergent.pml')
with open(pml_path, 'w') as f:
    f.write(pymol_template)
print(f"PyMOL template → {pml_path}")

## 6. Summary of generated files

After running this notebook, `data/subfamilies/` should contain
for each subfamily with ≥4 species:
- `SLC26AX.fa` — ungapped sequences
- `SLC26AX.aln.fa` — MAFFT alignment
- `SLC26AX.trim.fa` — trimAl -gt 0.1 (for analysis)
- `SLC26AX.trim05.fa` — trimAl -gt 0.5 (for visualization)
- `SLC26AX.nwk` — FastTree (LG model)